# Model Evaluation

This notebook regenerates the exact pipeline `main.py` runs
(`add_features` → `preprocess_data` → `train` → `evaluate`) and then adds
deeper diagnostic visuals (confusion matrix, ROC curve, precision-recall
curve, threshold tuning, feature importance) on top of the same model and
test split. Because every split in `src/` uses `random_state=42`, this
reproduces the identical train/test split used in
`04_model_training.ipynb`.

**Renamed from `04_evaluation.ipynb`.** The old version referenced a model
"trained in 03_modeling_training.ipynb" — a notebook that never existed
under that name (the actual training notebook is
`04_model_training.ipynb`). It also loaded `X_test.csv` / `y_test.csv`
from disk, which only works if a previous run happened to save them in a
compatible format. This version instead rebuilds the pipeline directly
from `src/`, so it's always self-contained and reproducible.

## 1. Imports

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, precision_recall_curve,
    recall_score, precision_score,
)

import joblib

from src.feature_engineering import add_features
from src.data_preprocessing import preprocess_data
from src.train_model import train
from src.evaluate_model import evaluate

import warnings
warnings.filterwarnings("ignore")

## 2. Rebuild the Pipeline & Train the Model

In [ ]:
raw_df = pd.read_csv("../data/raw/customer_churn_raw.csv")
raw_df.columns = raw_df.columns.str.strip().str.replace(" ", "_")

engineered_df = add_features(raw_df)
X_processed, y, raw_features = preprocess_data(engineered_df)

model, X_test, y_test = train(X_processed, y)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

## 3. Run the Production Evaluation — `src.evaluate_model.evaluate`

This is the exact function `main.py` calls, including the fix that aligns
`raw_features` rows to the correct test-set customers via `y_test.index`
rather than assuming the first N rows in file order correspond to the test
set. This is what makes `churn_risk_scores.csv` and the revenue-at-risk
figure trustworthy.

In [ ]:
results = evaluate(model, X_test, y_test, raw_features)
results["summary"]

## 4. Predictions for Diagnostics

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

## 5. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No Churn", "Churn"], yticklabels=["No Churn", "Churn"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.savefig("../reports/figures/confusion_matrix.png", bbox_inches="tight")
plt.show()

**How to read this:** false negatives (churn customers predicted as
non-churn, bottom-left cell) represent missed retention opportunities and
are the more business-costly error here; false positives (top-right cell)
mainly cost wasted retention-campaign spend. Compare the relative size of
these two cells to judge whether the model's current threshold (0.5) is
striking the right balance for your retention budget — see the threshold
tuning section below if not.

## 6. Classification Report

In [ ]:
print(classification_report(y_test, y_pred))

**How to read this:** with the split-then-SMOTE fix applied, expect
precision/recall/F1 for the churn class to be somewhat lower than the
~0.87–0.89 figures reported by the older (leaky) pipeline — that's the
expected, honest result of evaluating on a test set with no synthetic
rows.

## 7. ROC Curve & AUC

In [ ]:
roc_auc = roc_auc_score(y_test, y_prob)
fpr, tpr, _ = roc_curve(y_test, y_prob)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"Random Forest (AUC = {roc_auc:.3f})")
plt.plot([0,1], [0,1], "--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()

plt.savefig("../reports/figures/roc_curve.png", bbox_inches="tight")
plt.show()

**How to read this:** the further the curve bows toward the top-left
corner (away from the diagonal baseline), the better the model separates
churners from non-churners across all thresholds simultaneously — a
single summary of discrimination ability independent of any one cutoff.

## 8. Precision–Recall Curve

In [ ]:
precision, recall, _ = precision_recall_curve(y_test, y_prob)

plt.figure(figsize=(6,5))
plt.plot(recall, precision, color="blue")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")

plt.savefig("../reports/figures/precision_recall_curve.png", bbox_inches="tight")
plt.show()

**How to read this:** more informative than the ROC curve on this
imbalanced dataset (~93/7 split). A curve that stays high (close to 1.0
precision) across a wide range of recall values indicates the model can
catch a large share of churners without flooding the retention team with
false alarms.

## 9. Threshold Tuning for Churn Detection

The default decision threshold is 0.5. Lowering it trades precision for
recall — useful if missing a churner is costlier than a wasted retention
contact; raising it does the opposite.

In [ ]:
for t in np.arange(0.1, 1.0, 0.05):
    y_pred_t = (y_prob >= t).astype(int)
    prec = precision_score(y_test, y_pred_t, zero_division=0)
    rec = recall_score(y_test, y_pred_t)
    print(f"Threshold: {t:.2f} | Precision: {prec:.2f} | Recall: {rec:.2f}")

**How to use this table:** pick the threshold whose precision/recall
trade-off matches the actual cost ratio between a missed churner and a
wasted retention contact for your business, rather than defaulting to
0.5. Very low thresholds will push recall toward 1.0 at the cost of many
false positives; very high thresholds may stop predicting churn
altogether.

## 10. Feature Importance

In [ ]:
preprocessor = joblib.load("../models/preprocessor.pkl")
feature_names = preprocessor.get_feature_names_out()

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": model.feature_importances_,
}).sort_values(by="Importance", ascending=False)

top_features = feature_importance.head(20)

plt.figure(figsize=(8,6))
sns.barplot(x="Importance", y="Feature", data=top_features)
plt.title("Top 20 Feature Importances - Random Forest")

plt.savefig("../reports/figures/feature_importance.png", bbox_inches="tight")
plt.show()

## 11. Business Summary

Pulled directly from `results["summary"]` above — ROC-AUC, the churn risk
tier distribution, and the revenue-at-risk estimate, now computed against
correctly-aligned customers.

In [ ]:
summary = results["summary"]

print("ROC-AUC:", summary["roc_auc"])
print("\nChurn Risk Distribution:")
for k, v in summary["risk_distribution"].items():
    print(f"  {k:<25}: {v}%")
print(f"\nEstimated Revenue at Risk: {summary['revenue_at_risk']:,.2f} monetary units")

## 12. Conclusion & Model Selection

An end-to-end churn prediction pipeline was implemented, covering
preprocessing, feature engineering, leakage-safe class-imbalance handling
(split before SMOTE), model training, and detailed evaluation. Logistic
Regression, Decision Tree, and Random Forest were compared in
`04_model_training.ipynb` using business-critical metrics — recall and
ROC–AUC — on a test set free of synthetic rows.

**Random Forest** was selected as the production model for its superior
ability to identify churn customers while maintaining robust overall
performance, consistent with `src/train_model.py`. Threshold tuning above
shows how the operating point can be adjusted to match retention-budget
priorities, and feature importance adds interpretability for business
stakeholders rather than leaving the model as a black box.

The trained model and preprocessing pipeline are saved under `../models/`
for reproducibility and deployment, and `evaluate()` writes the
customer-level risk scores to `../data/processed/churn_risk_scores.csv`
for downstream retention workflows.